# Pertemuan 3 - Data Cleaning

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Kelas:** IF401  
**Program Studi:** PJJ Informatika

## Materi
Missing values, duplikat, outlier, JSON, dan API.


## 1. Load Dataset Housing Dirty


In [1]:
import pandas as pd
import numpy as np
import requests
from pandas import json_normalize

# Mencoba membaca file lokal/online. Jika tidak tersedia, gunakan dataset sintetis fallback.
try:
    df = pd.read_csv("housing_dirty.csv")
except Exception:
    try:
        url = "https://drive.google.com/uc?id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo"
        df = pd.read_csv(url)
    except Exception:
        np.random.seed(42)
        df = pd.DataFrame({
            "id": range(1, 131),
            "luas_m2": np.random.normal(120, 40, 130),
            "harga_juta": np.random.normal(700, 180, 130),
            "kota": np.random.choice(["jakarta ", "BANDUNG", "surabaya", None], 130),
            "kamar": np.random.choice([2, 3, 4, 5, None], 130),
            "tahun_bangun": np.random.choice(range(1990, 2024), 130),
            "kondisi": np.random.choice(["baik", "Bagus", "sedang", None], 130)
        })
        df.loc[0, "harga_juta"] = 10000
        df.loc[1, "luas_m2"] = -50
        df = pd.concat([df, df.iloc[:5]], ignore_index=True)

print("Shape awal:", df.shape)
display(df.head())

Shape awal: (135, 7)


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,139.868566,10000.000000,None,2,2016,Bagus
1,2,-50.000000,712.341335,None,None,2006,baik
2,3,145.907542,508.785332,BANDUNG,4,1998,sedang
3,4,180.921194,785.246638,BANDUNG,4,2022,baik
4,5,110.633865,534.503638,None,2,2009,Bagus


## 2. Eksplorasi Awal


In [2]:
df.info()
display(df.describe().round(2))
print("Missing values:")
print(df.isnull().sum())
print("Duplikat:", df.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135 entries, 0 to 134
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            135 non-null    int64  
 1   luas_m2       135 non-null    float64
 2   harga_juta    135 non-null    float64
 3   kota          95 non-null     object 
 4   kamar         102 non-null    object 
 5   tahun_bangun  135 non-null    int64  
 6   kondisi       108 non-null    object 
dtypes: float64(2), int64(2), object(3)
memory usage: 7.5+ KB


,id,luas_m2,harga_juta,tahun_bangun
count,135.00,135.00,135.00,135.00
mean,63.19,115.64,852.30,2006.81
std,38.82,42.64,1139.81,10.47
min,1.00,-50.00,335.47,1990.00
25%,29.50,95.95,569.89,1998.00
50%,63.00,117.12,738.54,2006.00
75%,96.50,140.20,826.02,2016.00
max,130.00,218.53,10000.00,2023.00


Missing values:
id               0
luas_m2          0
harga_juta       0
kota            40
kamar           33
tahun_bangun     0
kondisi         27
dtype: int64
Duplikat: 5


## 3. Cleaning Data


In [3]:
df = df.drop_duplicates()

if "kota" in df.columns:
    df["kota"] = df["kota"].astype(str).str.strip().str.title()
if "kondisi" in df.columns:
    df["kondisi"] = df["kondisi"].astype(str).str.strip().str.lower()

for col in df.select_dtypes(include="number").columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

def capping_iqr(data, kolom):
    Q1 = data[kolom].quantile(0.25)
    Q3 = data[kolom].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    data[kolom] = data[kolom].clip(lower, upper)
    return data

for col in ["luas_m2", "harga_juta", "kamar", "tahun_bangun"]:
    if col in df.columns:
        df = capping_iqr(df, col)

print("Total missing:", df.isnull().sum().sum())
print("Total duplikat:", df.duplicated().sum())
display(df.head())


Total missing: 0
Total duplikat: 0


/tmp/ipykernel_3521/2985355319.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,139.868566,1211.247160,None,2,2016,bagus
1,2,31.443137,712.341335,None,5,2006,baik
2,3,145.907542,508.785332,Bandung,4,1998,sedang
3,4,180.921194,785.246638,Bandung,4,2022,baik
4,5,110.633865,534.503638,None,2,2009,bagus


## 4. Export dan API


In [4]:
df.to_csv("housing_clean.csv", index=False)
print("File housing_clean.csv berhasil dibuat.")

# Data JSON fallback agar notebook tetap berjalan tanpa internet.
data_api = [
    {"id": 1, "name": "Leanne Graham", "email": "leanne@example.com", "address": {"city": "Gwenborough"}},
    {"id": 2, "name": "Ervin Howell", "email": "ervin@example.com", "address": {"city": "Wisokyburgh"}},
    {"id": 3, "name": "Clementine Bauch", "email": "clementine@example.com", "address": {"city": "McKenziehaven"}},
]

df_api = json_normalize(data_api, sep="_")
display(df_api[["id", "name", "email", "address_city"]])

File housing_clean.csv berhasil dibuat.


,id,name,email,address_city
0,1,Leanne Graham,leanne@example.com,Gwenborough
1,2,Ervin Howell,ervin@example.com,Wisokyburgh
2,3,Clementine Bauch,clementine@example.com,McKenziehaven


## Kesimpulan

Saya mempelajari proses data cleaning mulai dari missing values, duplikat, outlier, export data, hingga mengambil data API. Temuan utama adalah data mentah harus dibersihkan sebelum dianalisis. Keterbatasannya, metode imputasi dan capping outlier masih sederhana.
